# IPUMS batch exploration

This notebook reads the IPUMS International extract in `data/raw/ipumsi_00001.dat` using the DDI metadata from `data/raw/ipumsi_00001.xml`, then inspects the first batch of 1,000 records with `ipumspy` and `pandas`.

The goal is a quick structural check: file metadata, variable metadata, missingness, and a few column-level summaries.

In [1]:
from pathlib import Path
import warnings

import pandas as pd
from ipumspy import readers
from ipumspy.readers import CitationWarning

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

warnings.filterwarnings("ignore", category=CitationWarning)

ROOT = Path.cwd().resolve()
if not (ROOT / "data/raw/ipumsi_00001.dat").exists():
    ROOT = ROOT.parent.resolve()

DATA_DIR = ROOT / "data" / "raw"
DAT_PATH = DATA_DIR / "ipumsi_00001.dat"
DDI_PATH = DATA_DIR / "ipumsi_00001.xml"

DAT_PATH, DDI_PATH

(PosixPath('/home/aaron/bs/labor_market_structure_arg/data/raw/ipumsi_00001.dat'),
 PosixPath('/home/aaron/bs/labor_market_structure_arg/data/raw/ipumsi_00001.xml'))

In [2]:
ddi = readers.read_ipums_ddi(DDI_PATH)

extract_metadata = pd.DataFrame(
    [{
        "file_name": ddi.file_description.filename,
        "structure": ddi.file_description.structure,
        "encoding": ddi.file_description.encoding,
        "format": ddi.file_description.format,
        "variables": len(ddi.data_description),
        "samples": len(ddi.samples_description),
        "collection": ddi.ipums_collection,
        "doi": ddi.ipums_doi,
    }]
)
display(extract_metadata)

variable_metadata = pd.DataFrame(
    [
        {
            "name": desc.name,
            "label": desc.label,
            "start": desc.start + 1,
            "end": desc.end,
            "type": desc.vartype,
            "concept": desc.concept,
        }
        for desc in ddi.data_description[:20]
    ]
)
display(variable_metadata)

,file_name,structure,encoding,format,variables,samples,collection,doi
0,ipumsi_00001.dat,rectangular,iso-8859-1,fixed length fields,12,4,ipumsi,DOI:10.18128/D020.V7.7


,name,label,start,end,type,concept
0,COUNTRY,Country,1,3,integer,Technical Household Variables -- HOUSEHOLD
1,YEAR,Year,4,7,integer,Technical Household Variables -- HOUSEHOLD
2,SAMPLE,IPUMS sample identifier,8,16,integer,Technical Household Variables -- HOUSEHOLD
3,PERNUM,Person number,17,20,integer,Technical Person Variables -- PERSON
4,PERWT,Person weight,21,28,integer,Technical Person Variables -- PERSON
5,SEX,Sex,29,29,integer,Demographic Variables -- PERSON
6,EDATTAIN,"Educational attainment, international recode [general version]",30,30,integer,Education Variables -- PERSON
7,EDATTAIND,"Educational attainment, international recode [detailed version]",31,33,integer,Education Variables -- PERSON
8,OCC,"Occupation, unrecoded",34,37,integer,"Occupation, Industry Variables -- PERSON"
9,ISCO08A,"Occupation, ISCO-2008, 3-digit",38,40,integer,"Occupation, Industry Variables -- PERSON"


In [3]:
batch_reader = readers.read_microdata_chunked(
    ddi=ddi,
    filename=DAT_PATH,
    chunksize=1000,
)
batch = next(batch_reader)

batch.shape

(1000, 12)

In [4]:
display(batch.head(10))

print("dtypes")
display(batch.dtypes.to_frame(name="dtype"))

print("missingness (top 20)")
display(batch.isna().mean().sort_values(ascending=False).head(20).to_frame(name="missing_rate"))

numeric_cols = batch.select_dtypes(include="number").columns.tolist()
categorical_cols = batch.select_dtypes(include=["string", "object", "category"]).columns.tolist()

print("numeric columns:", numeric_cols[:10])
print("categorical columns:", categorical_cols[:10])

for column in categorical_cols[:3]:
    print(f"top values for {column}")
    display(
        batch[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
        .head(10)
    )

for column in numeric_cols[:3]:
    print(f"summary for {column}")
    display(batch[column].describe().to_frame(name=column))

,COUNTRY,YEAR,SAMPLE,PERNUM,PERWT,SEX,EDATTAIN,EDATTAIND,OCC,ISCO08A,INDGEN,IND
0,116,2019,116201901,1,10.0,2,1,120,6,611,10,11
1,116,2019,116201901,2,10.0,2,3,311,6,611,10,11
2,116,2019,116201901,3,10.0,1,1,120,6,611,10,11
3,116,2019,116201901,1,10.0,2,1,110,6,631,10,11
4,116,2019,116201901,1,10.0,1,1,120,9,933,80,522
5,116,2019,116201901,2,10.0,2,1,120,9,933,80,522
6,116,2019,116201901,3,10.0,1,2,212,9,933,80,522
7,116,2019,116201901,4,10.0,1,1,120,9,933,80,522
8,116,2019,116201901,5,10.0,2,1,120,99,999,0,999
9,116,2019,116201901,1,10.0,1,2,212,9,931,50,410


dtypes


,dtype
COUNTRY,Int64
YEAR,Int64
SAMPLE,Int64
PERNUM,Int64
PERWT,float64
SEX,Int64
EDATTAIN,Int64
EDATTAIND,Int64
OCC,Int64
ISCO08A,Int64


missingness (top 20)


,missing_rate
COUNTRY,0.0
YEAR,0.0
SAMPLE,0.0
PERNUM,0.0
PERWT,0.0
SEX,0.0
EDATTAIN,0.0
EDATTAIND,0.0
OCC,0.0
ISCO08A,0.0


...

,COUNTRY
count,1000.0
mean,116.0
std,0.0
min,116.0
25%,116.0
50%,116.0
75%,116.0
max,116.0


summary for YEAR


,YEAR
count,1000.0
mean,2019.0
std,0.0
min,2019.0
25%,2019.0
50%,2019.0
75%,2019.0
max,2019.0


summary for SAMPLE


,SAMPLE
count,1000.0
mean,116201901.0
std,0.0
min,116201901.0
25%,116201901.0
50%,116201901.0
75%,116201901.0
max,116201901.0


In [5]:
batch_reader = readers.read_microdata_chunked(
    ddi=ddi,
    filename=DAT_PATH,
    chunksize=20000,
)
batch = next(batch_reader)

batch.shape

(20000, 12)

In [7]:
batch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   COUNTRY    20000 non-null  Int64  
 1   YEAR       20000 non-null  Int64  
 2   SAMPLE     20000 non-null  Int64  
 3   PERNUM     20000 non-null  Int64  
 4   PERWT      20000 non-null  float64
 5   SEX        20000 non-null  Int64  
 6   EDATTAIN   20000 non-null  Int64  
 7   EDATTAIND  20000 non-null  Int64  
 8   OCC        20000 non-null  Int64  
 9   ISCO08A    20000 non-null  Int64  
 10  INDGEN     20000 non-null  Int64  
 11  IND        20000 non-null  Int64  
dtypes: Int64(11), float64(1)
memory usage: 2.0 MB


In [12]:
batch["COUNTRY"].value_counts(dropna=False).head(10)

COUNTRY
116    20000
Name: count, dtype: Int64